# 05 — Correlaciones: Clima vs Generación

**Objetivo:** Analizar la relación estadística entre la irradiancia (NASA) y la generación eléctrica en Cuba (ONEI).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path.cwd().parent / 'data'

# 1. Cargar los datos (usando los nombres exactos de los Parquet generados)
nasa = pd.read_parquet(DATA / 'clean' / 'nasa_power_limpio.parquet')
onei = pd.read_parquet(DATA / 'clean' / 'onei_generacion.parquet')

# 2. Agrupar NASA por año para poder cruzarlo con la tabla anual de la ONEI
nasa_anual = nasa.groupby('anio').agg({
    'irradiancia_kwh_m2': 'mean',
    'temperatura_c': 'mean',
    'humedad_pct': 'mean'
}).round(3).reset_index()

# 3. Unir ambas tablas usando la columna 'anio'
df_corr = onei.merge(nasa_anual, on='anio')
display(df_corr)

## 1. Matriz de Correlación (Heatmap)

Se utiliza el coeficiente de Spearman para evaluar relaciones no lineales.

In [ ]:
# Usamos las columnas exactas que existen en el DataFrame resultante
cols_numericas = ['gen_total_gwh', 'gen_fv_gwh', 'irradiancia_kwh_m2', 'temperatura_c', 'humedad_pct']
sub = df_corr[cols_numericas].dropna()

corr = sub.corr(method='spearman').round(3)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Matriz de Correlación: Generación vs Variables Climáticas')
plt.tight_layout()
plt.show()

## 2. Gráfico de Dispersión (Irradiancia vs Generación)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_corr, x='irradiancia_kwh_m2', y='gen_total_gwh', s=150, color='#f4a900')

for i, row in df_corr.iterrows():
    plt.text(row['irradiancia_kwh_m2'] + 0.05, row['gen_total_gwh'] - 100, str(row['anio']))

plt.title('Relación entre Irradiancia Solar y Generación Total (2020-2022)')
plt.xlabel('Irradiancia Promedio (kWh/m²/día)')
plt.ylabel('Generación Total (GW·h)')
plt.grid(alpha=0.3)
plt.show()